# AgentQuiver, final dataset preparation

Reads `../data/interim/` (output of the extraction notebook) and produces `../data/processed/`: **a clean interaction table, an item catalogue, two split protocols (per-developer leave-last-out and a global temporal cut), secondary signals and statistics**.

**No API calls**: the notebook runs in a few seconds. All parameters are in the first code cell.

Steps: 1) clean additions -> 2) noise exclusion -> 3) merge of equivalent tools across ecosystems -> 4) one item per (developer, tool), date = first adoption -> 5) k-core (item and developer thresholds) -> 6) splits -> 7) statistics -> 8) export.

In [ ]:
import re, json
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)

IN_DIR, OUT_DIR = Path("../data/interim"), Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Thresholds (k-core)
MIN_ITEM_DEVS = 3            # a tool must have been adopted by at least N developers
MIN_USER_INTERACTIONS = 5    # a developer must have adopted at least N tools (after the item filter)

# ---- Global temporal cut: train < T_VALID <= valid < T_TEST <= test
T_VALID, T_TEST = "2026-01-01", "2026-04-01"

# ---- Noise to exclude (automatically installed dependencies or build tools)
EXCLUDE_RE = re.compile(r"^(pypi:nvidia-|pypi:.*-cu\d+|npm:@types/)")
EXCLUDE_EXACT = {"pypi:pip", "pypi:setuptools", "pypi:wheel", "pypi:build", "pypi:twine",
                 "pypi:hatchling", "pypi:flit-core", "pypi:poetry-core"}

# ---- Same tool in two ecosystems (complete by hand if needed)
CANON = {
    "tool:openai": ["npm:openai", "pypi:openai"],
    "tool:anthropic": ["npm:@anthropic-ai/sdk", "pypi:anthropic"],
    "tool:google-genai": ["npm:@google/genai", "pypi:google-genai"],
    "tool:google-generativeai": ["npm:@google/generative-ai", "pypi:google-generativeai"],
    "tool:mcp-sdk": ["npm:@modelcontextprotocol/sdk", "pypi:mcp"],
    "tool:langchain": ["npm:langchain", "pypi:langchain"],
    "tool:langchain-core": ["npm:@langchain/core", "pypi:langchain-core"],
    "tool:langchain-openai": ["npm:@langchain/openai", "pypi:langchain-openai"],
    "tool:langchain-community": ["npm:@langchain/community", "pypi:langchain-community"],
    "tool:langgraph": ["npm:@langchain/langgraph", "pypi:langgraph"],
    "tool:langsmith": ["npm:langsmith", "pypi:langsmith"],
    "tool:transformers": ["pypi:transformers", "npm:@huggingface/transformers", "npm:@xenova/transformers"],
    "tool:huggingface-hub": ["pypi:huggingface-hub", "npm:@huggingface/hub"],
    "tool:chromadb": ["npm:chromadb", "pypi:chromadb"],
    "tool:qdrant": ["npm:@qdrant/js-client-rest", "pypi:qdrant-client"],
    "tool:pinecone": ["npm:@pinecone-database/pinecone", "pypi:pinecone-client", "pypi:pinecone"],
    "tool:ollama": ["npm:ollama", "pypi:ollama"],
    "tool:tiktoken": ["npm:tiktoken", "npm:js-tiktoken", "pypi:tiktoken"],
    "tool:onnxruntime": ["pypi:onnxruntime", "npm:onnxruntime-node", "npm:onnxruntime-web"],
    "tool:tensorflow": ["pypi:tensorflow", "npm:@tensorflow/tfjs"],
    "tool:llamaindex": ["pypi:llama-index", "npm:llamaindex"],
}
canon_of = {p: c for c, members in CANON.items() for p in members}

ev = pd.read_csv(IN_DIR / "dep_events.csv")
pk = pd.read_csv(IN_DIR / "packages.csv")
cur = pd.read_csv(IN_DIR / "current_deps.csv")
ev["date"] = pd.to_datetime(ev["date"], utc=True)
for c in ("is_ai",):
    ev[c] = ev[c].fillna(False).astype(bool)
    pk[c] = pk[c].fillna(False).astype(bool)
print(f"dep_events: {len(ev):,} rows | packages: {len(pk):,} | developers: {ev.developer.nunique():,}")

dep_events : 104,553 lignes | packages : 10,914 | développeurs : 576


In [ ]:
# 1) Clean additions
mask = (ev.action == "add") & (~ev.is_initial_commit) & ev.authored_by_dev & (ev.dep_kind != "dev") & (~ev.is_bulk)
ok = ev[mask].copy()
print(f"Clean additions: {len(ok):,} out of {len(ev):,} raw events ({len(ok)/len(ev):.0%})")

# 2) Noise to exclude
noise = ok.package_id.str.match(EXCLUDE_RE) | ok.package_id.isin(EXCLUDE_EXACT)
print(f"Excluded as noise: {noise.sum():,} additions ({ok[noise].package_id.nunique()} packages). Examples:",
      ", ".join(ok[noise].package_id.value_counts().head(8).index))
ok = ok[~noise].copy()

# 3) Merge of equivalent tools across ecosystems
ok["item"] = ok.package_id.map(canon_of).fillna(ok.package_id)
cur["item"] = cur.package_id.map(canon_of).fillna(cur.package_id)
merged = ok[ok.item.str.startswith("tool:")].groupby("item").package_id.nunique()
print(f"Merged tools present in the data: {(merged > 1).sum()} (out of {len(CANON)} rules)")

# Merge candidates NOT applied: same name in two ecosystems, among AI packages
ai_pk = pk[pk.is_ai & pk.package_id.isin(ok.package_id)].copy()
ai_pk["name"] = (ai_pk.package_id.str.split(":", n=1).str[1].str.lower()
                 .str.replace(r"^@[^/]+/", "", regex=True))
dups = ai_pk.groupby("name").package_id.apply(list)
dups = dups[dups.map(len) > 1]
todo = {k: v for k, v in dups.items() if not all(p in canon_of for p in v)}
print("\nMerge candidates to check by hand (add them to CANON if they really are the same tool):")
for k, v in todo.items():
    print(f"  {k:30s} {v}")

Ajouts propres : 10,115 sur 104,553 événements bruts (10%)
Exclus comme bruit : 159 ajouts (73 packages). Exemples : pypi:setuptools, npm:@types/three, npm:@types/node, npm:@types/react-dom, npm:@types/react, pypi:twine, pypi:wheel, pypi:build
Outils fusionnés présents dans les données : 13 (sur 21 règles)

Candidats de fusion à vérifier à la main (ajoute-les à CANON si ce sont bien les mêmes outils) :
  anthropic                      ['npm:@ai-sdk/anthropic', 'npm:@langchain/anthropic', 'pypi:anthropic']
  google-genai                   ['npm:@langchain/google-genai', 'pypi:google-genai']
  openai                         ['npm:@ai-sdk/openai', 'npm:@langchain/openai', 'npm:openai', 'pypi:openai']
  pi-agent-core                  ['npm:@earendil-works/pi-agent-core', 'npm:@mariozechner/pi-agent-core']
  react                          ['npm:@21st-sdk/react', 'npm:@assistant-ui/react']
  sdk                            ['npm:@agentclientprotocol/sdk', 'npm:@anthropic-ai/sdk', 'npm:@modelc

In [ ]:
# 4) One row per (developer, tool): date = first adoption
inter = (ok.groupby(["developer", "item"])
           .agg(date=("date", "min"), n_repos=("repo", "nunique"), n_events=("repo", "size"),
                ecosystems=("ecosystem", lambda s: "|".join(sorted(set(s)))))
           .reset_index())
in_use = set(zip(cur.developer, cur.item))
inter["still_in_use"] = [(d, i) in in_use for d, i in zip(inter.developer, inter.item)]
print(f"Interactions (developer, tool): {len(inter):,} | distinct tools: {inter.item.nunique():,}")

# Item catalogue (merges the metadata of the members of a merged tool)
pk["item"] = pk.package_id.map(canon_of).fillna(pk.package_id)
pk["first_release"] = pd.to_datetime(pk["first_release"], utc=True, errors="coerce")
pk["_prio"] = pk.package_id.str.startswith("pypi:").map({True: 0, False: 1})
pk = pk.sort_values(["item", "_prio"])
def agg_item(g):
    first = lambda col: next((v for v in g[col] if isinstance(v, str) and v), None)
    return pd.Series({
        "members": "|".join(g.package_id), "ecosystems": "|".join(sorted(set(g.ecosystem))),
        "summary": first("summary"), "keywords": first("keywords"), "classifiers": first("classifiers"),
        "repo_url": first("repo_url"), "is_ai": bool(g.is_ai.any()),
        "first_release": g.first_release.min(),
    })
items_meta = pk[pk.item.isin(inter.item.unique())].groupby("item").apply(agg_item, include_groups=False).reset_index()
print(f"Catalogue: {len(items_meta):,} tools | flagged AI: {items_meta.is_ai.sum():,}")

Interactions (développeur, outil) : 8,427 | outils distincts : 3,571
Catalogue : 3,571 outils | marqués IA : 213


In [ ]:
# 5) k-core: item and developer thresholds, repeated until stable
core = inter.copy()
n_before = (core.developer.nunique(), core.item.nunique(), len(core))
while True:
    ic = core.groupby("item").developer.nunique()
    core = core[core.item.isin(ic[ic >= MIN_ITEM_DEVS].index)]
    uc = core.groupby("developer").size()
    new = core[core.developer.isin(uc[uc >= MIN_USER_INTERACTIONS].index)]
    if len(new) == len(core):
        break
    core = new
assert len(core) > 0, "Empty k-core: lower MIN_ITEM_DEVS or MIN_USER_INTERACTIONS"
print(f"Before k-core: {n_before[0]} developers, {n_before[1]:,} tools, {n_before[2]:,} interactions")
print(f"After k-core: {core.developer.nunique()} developers, {core.item.nunique():,} tools, {len(core):,} interactions")

# Integer identifiers
core = core.sort_values(["developer", "date", "item"]).reset_index(drop=True)
user_ids = {d: i for i, d in enumerate(sorted(core.developer.unique()))}
item_ids = {t: i for i, t in enumerate(sorted(core.item.unique()))}
core["user_id"] = core.developer.map(user_ids)
core["item_id"] = core.item.map(item_ids)
core["ts"] = (core.date.astype("int64") // 10**9).astype("int64")
ai_items = set(items_meta[items_meta.is_ai].item)
core["is_ai"] = core.item.isin(ai_items)

# 6) Splits
# (a) Per-developer leave-last-out: last tool = test, second to last = valid, the rest = train
pos = core.groupby("user_id").cumcount()
size = core.groupby("user_id")["item"].transform("size")
core["split_loo"] = np.where(pos == size - 1, "test", np.where(pos == size - 2, "valid", "train"))
# (b) Global temporal cut
tv, tt = pd.Timestamp(T_VALID, tz="UTC"), pd.Timestamp(T_TEST, tz="UTC")
core["split_temporal"] = np.where(core.date >= tt, "test", np.where(core.date >= tv, "valid", "train"))
# Item already seen in train? (to spot cold start)
for col in ("split_loo", "split_temporal"):
    seen = set(core.loc[core[col] == "train", "item"])
    core[f"item_in_train_{col.split('_', 1)[1]}"] = core.item.isin(seen)
core.head()

Avant k-core : 492 développeurs, 3,571 outils, 8,427 interactions
Après k-core : 322 développeurs, 607 outils, 4,611 interactions


,developer,item,date,n_repos,n_events,ecosystems,still_in_use,user_id,item_id,ts,is_ai,split_loo,split_temporal,item_in_train_loo,item_in_train_temporal
0,1411430556,npm:axios,2023-04-02 10:11:13+00:00,2,2,npm,True,0,156,1680430273,False,train,train,True,True
1,1411430556,npm:moment,2023-04-02 10:11:13+00:00,2,2,npm,True,0,282,1680430273,False,train,train,True,True
2,1411430556,npm:vue-router,2023-04-02 10:11:13+00:00,1,1,npm,True,0,411,1680430273,False,train,train,True,True
3,1411430556,npm:vuex,2023-04-02 10:11:13+00:00,2,2,npm,True,0,412,1680430273,False,train,train,True,True
4,1411430556,npm:nprogress,2023-05-08 09:03:34+00:00,1,1,npm,False,0,301,1683536614,False,train,train,True,True


In [ ]:
# 7) Statistics (to reuse in the project report)
U, I, N = core.user_id.nunique(), core.item_id.nunique(), len(core)
per_user, per_item = core.groupby("user_id").size(), core.groupby("item_id").user_id.nunique()
stats = {
    "developers": U, "items": I, "interactions": N, "sparsity_%": round(100 * (1 - N / (U * I)), 2),
    "interactions_per_dev_median": float(per_user.median()), "interactions_per_dev_mean": round(per_user.mean(), 1),
    "devs_per_item_median": float(per_item.median()),
    "ai_items": int(core[core.is_ai].item_id.nunique()), "ai_interactions": int(core.is_ai.sum()),
    "date_min": str(core.date.min().date()), "date_max": str(core.date.max().date()),
    "still_in_use_%": round(100 * core.still_in_use.mean(), 1),
}
print("=== Final dataset ===")
for k, v in stats.items():
    print(f"  {k:32s} {v}")

def split_report(col, min_train=3):
    r = {}
    for s in ("train", "valid", "test"):
        sub = core[core[col] == s]
        r[f"{s}_interactions"] = len(sub)
        r[f"{s}_ai_interactions"] = int(sub.is_ai.sum())
    tr = core[core[col] == "train"].groupby("user_id").size()
    te = core[core[col] == "test"].groupby("user_id").size()
    eval_users = set(tr[tr >= min_train].index) & set(te.index)
    ai_te = core[(core[col] == "test") & core.is_ai].user_id
    r["eval_users"] = len(eval_users)
    r["eval_users_with_ai_target"] = len(eval_users & set(ai_te))
    te_rows = core[core[col] == "test"]
    r["test_cold_item_share_%"] = round(100 * (~te_rows[f"item_in_train_{col.split('_', 1)[1]}"]).mean(), 1) if len(te_rows) else None
    return r

reports = {"leave_last_out": split_report("split_loo"), "temporal": split_report("split_temporal")}
print("\n=== Split protocols ===")
print(pd.DataFrame(reports).to_string())
print(f"\n(temporal: train < {T_VALID} <= valid < {T_TEST} <= test)")

print("\n=== Most adopted tools ===")
top = (core.groupby("item").agg(devs=("user_id", "nunique"), is_ai=("is_ai", "first"))
          .sort_values("devs", ascending=False).head(20))
print(top.to_string())

=== Jeu de données final ===
  developers                       322
  items                            607
  interactions                     4611
  sparsity_%                       97.64
  interactions_per_dev_median      11.0
  interactions_per_dev_mean        14.3
  devs_per_item_median             5.0
  ai_items                         43
  ai_interactions                  355
  date_min                         2014-06-09
  date_max                         2026-11-18
  still_in_use_%                   91.0

=== Protocoles de découpe ===
                           leave_last_out  temporal
train_interactions                 3967.0    1970.0
train_ai_interactions               292.0      95.0
valid_interactions                  322.0     589.0
valid_ai_interactions                27.0      62.0
test_interactions                   322.0    2052.0
test_ai_interactions                 36.0     198.0
eval_users                          322.0      98.0
eval_users_with_ai_target            

In [ ]:
# 8) Export
core_cols = ["user_id", "item_id", "developer", "item", "ts", "date", "n_repos", "n_events", "ecosystems",
             "still_in_use", "is_ai", "split_loo", "split_temporal", "item_in_train_loo", "item_in_train_temporal"]
core[core_cols].to_csv(OUT_DIR / "interactions.csv", index=False)
inter.assign(is_ai=inter.item.isin(ai_items)).to_csv(OUT_DIR / "interactions_all_no_kcore.csv", index=False)

items_out = (items_meta[items_meta.item.isin(item_ids)]
             .assign(item_id=lambda d: d.item.map(item_ids))
             .merge(core.groupby("item").agg(n_devs=("user_id", "nunique"), n_interactions=("user_id", "size")).reset_index(), on="item")
             .sort_values("item_id"))
items_out.to_csv(OUT_DIR / "items.csv", index=False)
pd.DataFrame({"developer": list(user_ids), "user_id": list(user_ids.values())}).to_csv(OUT_DIR / "users.csv", index=False)

# Secondary signals restricted to the retained developers
for name, tcol in (("stars", "starred_at"), ("forks", "forked_at")):
    df = pd.read_csv(IN_DIR / f"{name}.csv")
    df = df[df.developer.isin(user_ids)].assign(user_id=lambda d: d.developer.map(user_ids))
    df[tcol] = pd.to_datetime(df[tcol], utc=True)
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name}.csv: {len(df):,} rows for {df.user_id.nunique()} developers")

json.dump({"params": {"MIN_ITEM_DEVS": MIN_ITEM_DEVS, "MIN_USER_INTERACTIONS": MIN_USER_INTERACTIONS,
                      "T_VALID": T_VALID, "T_TEST": T_TEST}, "stats": stats, "splits": reports, "canon": CANON},
          open(OUT_DIR / "dataset_info.json", "w"), indent=2, ensure_ascii=False, default=str)
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:34s} {f.stat().st_size/1e6:7.2f} MB")

stars.csv : 14,310 lignes pour 243 développeurs
forks.csv : 2,948 lignes pour 234 développeurs
dataset_info.json                     0.00 Mo
forks.csv                             0.26 Mo
interactions.csv                      0.53 Mo
interactions_all_no_kcore.csv         0.64 Mo
items.csv                             0.23 Mo
stars.csv                             3.22 Mo
users.csv                             0.01 Mo


## Dictionary of `data/processed/`

| File | Contents |
|---|---|
| **`interactions.csv`** | one row per (developer, tool) after k-core: `user_id`, `item_id`, `ts` (adoption), `n_repos`, `still_in_use`, `is_ai`, **`split_loo`** and **`split_temporal`** (train / valid / test), `item_in_train_*` (False = cold start) |
| `items.csv` | catalogue of the retained tools: summary, keywords, repository, `first_release`, `is_ai`, `n_devs`, `members` (merged packages) |
| `users.csv` | mapping between developer and `user_id` |
| `stars.csv`, `forks.csv` | secondary signal of the same developers, with `user_id` |
| `interactions_all_no_kcore.csv` | all interactions before k-core (for analyses or cold start) |
| `dataset_info.json` | parameters, statistics and merge rules used |

**Interaction date**: first adoption of the package by the developer, across all their repositories. **Order within the same instant**: additions from the same commit share the same `ts`; their relative order is arbitrary (alphabetical sort).

**Two protocols**: `split_loo` (standard in sequential recommendation, more evaluable users) and `split_temporal` (more realistic, avoids mixing past and future across developers). Report both.